# Gresho-Chan Vortex (2D)

This notebook runs the Gresho-Chan vortex benchmark: a rotating vortex whose centripetal acceleration is exactly balanced by its own pressure gradient, so the analytic solution is steady -- velocities and pressures should not drift over the run. Any drift visible in the plot below is scheme error, not physics.

Like `08-hydrostatic.ipynb`, this is a `particlePlot` (2D field view) case, so plotting calls `buildFieldPlotter`/`refreshFieldPlotter` directly (from `warpSPH.cases.plotting`) on `GRESHO_FIELDS` (exported from `warpSPH.cases.greshoVortex`) rather than going through `greshoVortexCase.setupPlot`/`updatePlot`, which go through `openWindow`/`pumpEvents` and do not live-update reliably inside a Jupyter cell in this environment.

`greshoVortexCase` has no `timestep` hook and leaves `dt` unset in its defaults, so -- like Sod -- `sampleGreshoVortex` itself picks the CFL-derived `dt` during IC construction, and the loop below is a fixed `range(nSteps)`.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/09-Gresho_Chan_Vortex.gif)


In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.greshoVortex import greshoVortexCase, GRESHO_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `09-gresho-chan-vortex.py`, made explicit and editable here.
# `greshoVortexCase.defaults`/`.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=greshoVortexCase.name, scheme=greshoVortexCase.scheme,
                params=dict(greshoVortexCase.params)) \
    .merged(**greshoVortexCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=200,
    dim=2,
    L=1.0,

    # --- time stepping ---------------------------------------------------
    tLimit=3.0,
    # No `dt` here -- `sampleGreshoVortex` leaves the CFL-derived value from
    # `computeTimestep` in `config.dt`, and there is no `timestep` hook to
    # re-pick it, so it stays fixed for the whole run.

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- Gresho-Chan's own knobs (none beyond the shared compressible ones) --
    params=dict(
        markerSize=4,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`greshoVortexCase.buildSystem` -> `sampleGreshoVortex`), not re-derived
# here.
ctx = buildContext(greshoVortexCase, spec)
greshoVortexCase.configureScheme(ctx)
system = greshoVortexCase.buildSystem(ctx)
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(GRESHO_FIELDS), not greshoVortexCase.setupPlot --
# see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, GRESHO_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = greshoVortexCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=greshoVortexCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = greshoVortexCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, GRESHO_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=greshoVortexCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
